# 0054: one programs round (between two passes)

**Runtime:** an A100, as the first round. **No model API is called; nothing needs a key.** A few minutes: loading
the 450M is most of it.

The stack's later pass at home handed its programs seat prompts Colab had never seen. This notebook writes the
450M's program for each of them (`prompts.jsonl` in, `programs.jsonl` out, same folder), with the first round's
code, base and adapter. Claude Code replays them at home.

**Runtime > Run all.** It checks itself, then disconnects.

In [ ]:
import hashlib, json, os, subprocess, sys, time
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cubbyllm'
REPO  = '/content/CubbyLLM'
FIRST = json.load(open(f'{DRIVE}/system_eval/run_0054/frozen_gpu.json'))   # the first round: its commit, its adapter
ROUND = f'{DRIVE}/system_eval/run_0055_after'                                  # prompts.jsonl in; programs.jsonl out
TOKENIZER = f'{DRIVE}/token_cache_base/tokenizer/grillcheese_bbpe128k.json'
CKPT      = f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt'           # the programs adapter's base
ADAPTER   = f'{DRIVE}/emitter/cubby450m_v12e_w_step_cont'                  # the serving programs seat
for p in (TOKENIZER, CKPT, f'{ADAPTER}/emitter_lora.safetensors', f'{ROUND}/prompts.jsonl'):
    assert os.path.exists(p), p

def sh(*cmd):
    subprocess.run(cmd, check=True)

if not os.path.exists(REPO):
    sh('git', 'clone', '-q', 'https://github.com/Grillcheese-AI/CubbyLLM.git', REPO)
sh('git', '-C', REPO, 'fetch', '-q', 'origin')
sh('git', '-C', REPO, 'checkout', '-q', FIRST['commit'])                  # the code that wrote the first round's programs
sh(sys.executable, '-m', 'pip', '-q', 'install', 'tokenizers', 'safetensors', 'numpy')

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()[:16]

commit  = subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip()
adapter = sha(f'{ADAPTER}/emitter_lora.safetensors')
card    = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
assert commit.startswith(FIRST['commit']), (commit, FIRST['commit'])
assert adapter == FIRST['programs_adapter'], ('not the first round\'s programs adapter', adapter, FIRST['programs_adapter'])
n_prompts = sum(1 for x in open(f'{ROUND}/prompts.jsonl', encoding='utf-8') if x.strip())
print('commit', commit[:7], '| adapter', adapter, '| card', card, '| prompts', n_prompts)

In [ ]:
cmd = [sys.executable, '-u', 'validation/system_eval/gpu.py', 'programs', '--prompts', f'{ROUND}/prompts.jsonl',
       '--out', f'{ROUND}/programs.jsonl', '--ckpt', CKPT, '--adapter', ADAPTER, '--tokenizer', TOKENIZER]
with open(f'{ROUND}/round.log', 'a', encoding='utf-8', buffering=1) as f:
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end=''); f.write(line)
    assert p.wait() == 0, f'programs failed ({p.returncode})'
rows = [json.loads(x) for x in open(f'{ROUND}/programs.jsonl', encoding='utf-8') if x.strip()]
want = {json.loads(x)['prompt'] for x in open(f'{ROUND}/prompts.jsonl', encoding='utf-8') if x.strip()}
assert {r['prompt'] for r in rows} == want, (len(rows), len(want))
for r in rows:
    print(repr(r['prompt'][-90:]), '->', repr(r['program'][:240]))
json.dump({'commit': commit[:7], 'programs_adapter': adapter, 'card': card, 'date': time.strftime('%Y-%m-%d'),
           'prompts': len(want), 'programs': len(rows)}, open(f'{ROUND}/frozen_gpu.json', 'w'), indent=1)
print('DONE:', len(rows), 'programs for', len(want), 'prompts')

In [ ]:
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()